# Soil Moisture and Recharge Modeling Notebook

This notebook is the preceding workflow to the `LandslideProbability` modeling notebook.
It computes daily root-zone water balance and recharge for a storm event of interest,
producing the spatially distributed recharge grids (.asc) that serve as input to the
landslide probability model.

The workflow combines three Landlab ecohydrology components:

- **`Radiation`** — computes spatially distributed net radiation from daily min/max
  temperature, latitude, albedo, and day-of-year.
- **`PotentialEvapotranspiration`** — estimates PET using the Priestley-Taylor method
  driven by radiation and temperature.
- **`SoilMoisture`** — a single-layer, depth-averaged root-zone water balance model
  driven by daily effective rainfall (after runoff abstraction) and PET, producing
  soil moisture and root-zone leakage (recharge) at each cell.

The daily time loop runs over a 7-day window centred on the peak rainfall day of the
event (5 days before peak + 2 days after). The cell-wise maximum across all days is computed and saved as an ESRI ASCII grid
for direct input to `LandslideProbability`.

**To model a different event**, replace the precipitation and temperature rasters in
Section 5 with the dates of interest and update `current_time` in Section 4 to the
corresponding day-of-year fraction. Run the notebook from top to bottom.

### References
Jimenez, H. N., Istanbulluoglu, E., Gorum, T., Stanley, T. A., Amatya, P. M., Tanyas, H.,
Demirel, M. C., Akgun, A., and Bozkurt, D.: Modeling the combined effects of the 2023
Türkiye-Syria Earthquake and an Atmospheric River event on landslide hazard, EGUsphere
[preprint], https://doi.org/10.5194/egusphere-2025-3011, 2025.

Strauch, R., Istanbulluoglu, E., Nudurupati, S., Bandaragoda, C., Gasparini, N., Tucker, G.
(2018). A hydroclimatological approach to predicting regional landslide probability using
Landlab. *Earth Surface Dynamics*, 6(1), 49–75. https://doi.org/10.5194/esurf-6-49-2018

Nudurupati, S. S., Istanbulluoglu, E., Tucker, G. E., Gasparini, N. M., Hobley, D. E. J.,
Hutton, E. W. H., et al. (2023). On transient semi-arid ecosystem dynamics using Landlab:
Vegetation shifts, topographic refugia, and response to climate. *Water Resources Research*,
59, e2021WR031179. https://doi.org/10.1029/2021WR031179

---
## To run this notebook
Click in each cell and press **Shift + Enter**, or use **Cell → Run All** from the menu.
All required input files must be present in the paths specified in **Section 2**.


## 1. Import Libraries

Standard Python libraries plus Landlab utilities and three custom ecohydrology
components supplied with this repository:
`soil_moisture_dynamics.py`, `radiation.py`, and
`potential_evapotranspiration_field.py`.


In [1]:
import numpy as np
import time

from landlab.io import read_esri_ascii, write_esri_ascii
from landlab.grid.mappers import map_node_to_cell
from landlab.components import FlowAccumulator, SinkFillerBarnes
from landlab.components.flow_accum import find_drainage_area_and_discharge

from soil_moisture_dynamics import SoilMoisture
from radiation import Radiation
from potential_evapotranspiration_field import PotentialEvapotranspiration

import warnings
warnings.simplefilter("ignore", DeprecationWarning)

st = time.time()


## 2. Input / Output File Paths

All raster inputs are ESRI ASCII grids (.asc). Adjust these paths to match your local
directory structure.  Soil and vegetation parameter rasters are organized in the `data_folder`. Daily precipitation inputs (IMERG V7) and (ERA5) temperature inputs are suborganized in date labeled folders. **To model a different event, replace the event folder filename to match your desired storm**


In [2]:
data_folder   = 'soilmoisture_input/'    # soil/vegetation parameters
event_folder  = data_folder + 'March 2023/'    # daily climate forcing
output_folder = 'recharge/'   # destination for recharge output grids


## 3. Build the Landlab Raster Model Grid

The DEM defines the model domain. No-data cells (−9999) are set to closed boundary
nodes and the minimum-elevation core node is designated the watershed outlet.
Min and max elevation are stored for reference.


In [3]:
(grid, Z) = read_esri_ascii(data_folder + 'landslide_run_dem.asc',
                            name='topographic__elevation')

grid.set_nodata_nodes_to_closed(Z, -9999.)

outlet_id = grid.core_nodes[np.argmin(grid.at_node['topographic__elevation'][grid.core_nodes])]
grid.set_watershed_boundary_condition_outlet_id(outlet_id, Z)

Zmin = np.min(grid.at_node['topographic__elevation'][grid.core_nodes])
Zmax = np.max(grid.at_node['topographic__elevation'][grid.core_nodes])

print(f"Outlet node ID       : {outlet_id}")
print(f"Outlet elevation (m) : {grid.at_node['topographic__elevation'][outlet_id]:.2f}")
print(f"Number of core nodes : {grid.number_of_core_nodes}")
print(f"Watershed area (km2) : {grid.number_of_core_nodes * grid.dx * grid.dy / 1e6:.2f}")


Outlet node ID       : 12264
Outlet elevation (m) : 336.00
Number of core nodes : 2960569
Watershed area (km2) : 23980.61


## 4. Event Parameters and Output Accumulators

`Latitude` and `current_time` (fractional day-of-year, where 1.0 = Jan 1) are set
to correspond to the event start date. For the March 2023 AR event in Türkiye,
the 7-day window runs March 10–16, 2023; `current_time = 0.20` initialises the
model at roughly the March 13 timestamp.

Albedo is assumed constant at 0.2. The `SoilMoisture` component uses a bare-soil
fraction (`f_bare`) of 0.7, consistent with the semi-arid to sub-humid vegetation
cover of the study area.

Output lists accumulate per-day spatial arrays of recharge, runoff, soil moisture,
and ET for the full event window.

> **To model a different event**: update `Latitude`, `current_time`, and the raster
> filenames in Section 5 to match the dates and location of interest.


In [4]:
# ── Event configuration ───────────────────────────────────────────────────────
Latitude    = 38       # decimal degrees N (Türkiye study area)
Albedo      = 0.2      # surface albedo (assumed constant)
current_time = 0.2   # fractional day-of-year for event start (March 10, 2023)

# ── Per-day output accumulators ───────────────────────────────────────────────
recharge_arrays      = []   # root-zone leakage (mm/day) at each node
runoff_arrays        = []   # surface runoff (mm/day) at each node
soil_moisture_arrays = []   # volumetric soil moisture saturation fraction at each node
ET_arrays            = []   # actual evapotranspiration (mm/day) at each node

# ── Regional-average time series (diagnostic) ────────────────────────────────
P_REG   = []   # spatial mean precipitation (mm/day)
SM_REG  = []   # spatial mean soil moisture (vol. fraction)
RE_REG  = []   # spatial mean recharge (mm/day)
PET_REG = []   # spatial mean PET (mm/day)
AET_REG = []   # spatial mean AET (mm/day)


## 5. Soil Parameters

Four soil hydraulic properties are loaded from pre-processed rasters. All water content values are stored as integers scaled
by 0.0001 (i.e., original units of 0.0001 m³/m³); dividing by 0.0001 recovers
volumetric water content in m³/m³.

- **Field capacity** (`WCpF2`): water content at pF 2 (−10 kPa).
- **Saturated water content** (`WCsat`): total porosity, used as the upper bound
  for soil moisture and for computing effective pore volume in the runoff model.
- **Wilting point** (`WCpF4`): water content at pF 4.2 (−1500 kPa).
- **Soil thickness** (`hs`): required to close the water balance at each node.

Initial soil saturation fraction is estimated assuming soils begin at 35% of the
available water capacity above wilting point, representing typical early-spring
antecedent conditions.


In [5]:
# Soil thickness
(_, hs) = read_esri_ascii(data_folder + 'soil__thickness.asc', name='soil__thickness')
_ = grid.add_field('soil__thickness', hs, at='node', clobber=True)
grid.set_nodata_nodes_to_closed(hs, -9999.)

# Field capacity (m3/m3)
(_, WCpF2) = read_esri_ascii(data_folder + 'field_capacity.asc', name='field__capacity')
_ = grid.add_field('field_capacity', WCpF2 * 0.0001, at='node', clobber=True)
fc = grid.at_node['field_capacity']

# Saturated water content / porosity (m3/m3)
(_, WCsat) = read_esri_ascii(data_folder + 'saturated_water_content.asc',
                              name='saturated_water__content')
_ = grid.add_field('saturated_water__content', WCsat * 0.0001, at='node', clobber=True)
n = grid.at_node['saturated_water__content']
Porosity = map_node_to_cell(grid, 'saturated_water__content')  # used in runoff model

# Wilting point (m3/m3)
(_, WCpF4) = read_esri_ascii(data_folder + 'wilting_point.asc', name='wilting__point')
_ = grid.add_field('wilting_point', WCpF4 * 0.0001, at='node', clobber=True)
wp = grid.at_node['wilting_point']

# Initial soil saturation fraction: 35% of available water capacity above wilting point
Saturation_fraction = (0.35 * (fc - wp) + wp) / n
_ = grid.add_field('soil_moisture__initial_saturation_fraction',
                   Saturation_fraction, at='node', clobber=True)
grid['cell']['soil_moisture__initial_saturation_fraction'] = map_node_to_cell(
    grid, 'soil_moisture__initial_saturation_fraction')

print(f"Mean initial saturation fraction: {Saturation_fraction[grid.core_nodes].mean():.3f}")


Mean initial saturation fraction: 0.532


## 6. Vegetation Parameters

Two vegetation inputs are required by `SoilMoisture` and `PotentialEvapotranspiration`:

- **Plant functional type (PFT)**: integer class raster (e.g. grass, shrub, tree,
  bare). Used to assign vegetation-class-specific runoff curve numbers and
  interception parameters within the component.
- **Leaf area index (LAI)**: from MODIS or equivalent. Values > 1.44 are capped at
  1.44 (full grass-equivalent cover per the Priestley-Taylor reference equation).
  Vegetation cover fraction `Vt = LAI / 1.5` is derived from LAI following the
  approach in Nudurupati et al. (2023).

Both fields are mapped from nodes to cells before being passed to the components.


In [6]:
# Plant functional type
(_, PFT) = read_esri_ascii(data_folder + 'vegetation__plant_functional_type.asc', name='PFT')
_ = grid.add_field('vegetation__plant_functional_type', PFT, at='node', clobber=True)
grid['cell']['vegetation__plant_functional_type'] = map_node_to_cell(
    grid, 'vegetation__plant_functional_type').astype(int)

# Leaf area index — cap at 1.44 (full grass-equivalent cover)
(_, Lai) = read_esri_ascii(data_folder + 'lai_cleaned.asc', name='leaf_area__index')
_ = grid.add_field('vegetation__live_leaf_area_index', Lai, at='node', clobber=True)
grid['cell']['vegetation__live_leaf_area_index'] = map_node_to_cell(
    grid, 'vegetation__live_leaf_area_index')

for i in range(len(Lai)):
    if Lai[i] > 1.44:
        Lai[i] = 1.44

# Vegetation cover fraction derived from LAI
Vt = Lai / 1.5
_ = grid.add_field('vegetation__cover_fraction', Vt, at='node', clobber=True)
grid['cell']['vegetation__cover_fraction'] = map_node_to_cell(grid, 'vegetation__cover_fraction')


## 7. Daily Climate Forcing

Daily precipitation (IMERG V7, mm) and ERA5 min/max temperature (°C) are loaded for
each day of the event window and organised into lists for iteration in the time loop.

The example below covers the March 2023 AR event (March 10–16, 2023). **To model a
different event, replace the filenames and/or date range.** The number of days in
`rainfall_arrays`, `tempmin_arrays`, and `tempmax_arrays` must all match.


In [7]:
# Daily precipitation (IMERG V7, mm)
(_, P1) = read_esri_ascii(event_folder + 'IMERGV7PrecipitationMM20230310.asc', name='Precipitation')
(_, P2) = read_esri_ascii(event_folder + 'IMERGV7PrecipitationMM20230311.asc', name='Precipitation')
(_, P3) = read_esri_ascii(event_folder + 'IMERGV7PrecipitationMM20230312.asc', name='Precipitation')
(_, P4) = read_esri_ascii(event_folder + 'IMERGV7PrecipitationMM20230313.asc', name='Precipitation')
(_, P5) = read_esri_ascii(event_folder + 'IMERGV7PrecipitationMM20230314.asc', name='Precipitation')
(_, P6) = read_esri_ascii(event_folder + 'IMERGV7PrecipitationMM20230315.asc', name='Precipitation')
(_, P7) = read_esri_ascii(event_folder + 'IMERGV7PrecipitationMM20230316.asc', name='Precipitation')

# Daily minimum temperature (ERA5, °C)
(_, Tmin1) = read_esri_ascii(event_folder + 'ERA5TemperatureMinC20230310.asc', name='Temperature_min')
(_, Tmin2) = read_esri_ascii(event_folder + 'ERA5TemperatureMinC20230311.asc', name='Temperature_min')
(_, Tmin3) = read_esri_ascii(event_folder + 'ERA5TemperatureMinC20230312.asc', name='Temperature_min')
(_, Tmin4) = read_esri_ascii(event_folder + 'ERA5TemperatureMinC20230313.asc', name='Temperature_min')
(_, Tmin5) = read_esri_ascii(event_folder + 'ERA5TemperatureMinC20230314.asc', name='Temperature_min')
(_, Tmin6) = read_esri_ascii(event_folder + 'ERA5TemperatureMinC20230315.asc', name='Temperature_min')
(_, Tmin7) = read_esri_ascii(event_folder + 'ERA5TemperatureMinC20230316.asc', name='Temperature_min')

# Daily maximum temperature (ERA5, °C)
(_, Tmax1) = read_esri_ascii(event_folder + 'ERA5TemperatureMaxC20230310.asc', name='Temperature_max')
(_, Tmax2) = read_esri_ascii(event_folder + 'ERA5TemperatureMaxC20230311.asc', name='Temperature_max')
(_, Tmax3) = read_esri_ascii(event_folder + 'ERA5TemperatureMaxC20230312.asc', name='Temperature_max')
(_, Tmax4) = read_esri_ascii(event_folder + 'ERA5TemperatureMaxC20230313.asc', name='Temperature_max')
(_, Tmax5) = read_esri_ascii(event_folder + 'ERA5TemperatureMaxC20230314.asc', name='Temperature_max')
(_, Tmax6) = read_esri_ascii(event_folder + 'ERA5TemperatureMaxC20230315.asc', name='Temperature_max')
(_, Tmax7) = read_esri_ascii(event_folder + 'ERA5TemperatureMaxC20230316.asc', name='Temperature_max')

# Organise into lists for the daily time loop (Section 9)
rainfall_arrays = [P1, P2, P3, P4, P5, P6, P7]
tempmin_arrays  = [Tmin1, Tmin2, Tmin3, Tmin4, Tmin5, Tmin6, Tmin7]
tempmax_arrays  = [Tmax1, Tmax2, Tmax3, Tmax4, Tmax5, Tmax6, Tmax7]


## 8. Instantiate Ecohydrology Components

The three Landlab ecohydrology components are instantiated once before the time loop.
Their internal state (radiation, PET, soil moisture) is updated each iteration via
`.update()`. Key parameter choices:

- `method='Grid'` for `Radiation`: computes spatially distributed net radiation
  using the gridded Tmin/Tmax fields rather than a single point value.
- `clearsky_turbidity=1.0`: clear-sky atmospheric turbidity (Linke turbidity factor).
- `method='PriestleyTaylor'` for PET: uses the Priestley-Taylor α coefficient (default
  α = 1.26) applied to net radiation.
- `runon=0`: no lateral water redistribution between cells in `SoilMoisture`.
- `f_bare=0.7`: fraction of bare soil, controlling soil evaporation relative to
  transpiration. Set to 0.7 consistent with semi-arid to sub-humid conditions.

An initial `rainfall__daily_depth` cell field of ones is required at instantiation
by `SoilMoisture`; it is overwritten each day in the time loop.


In [8]:
grid['cell']['rainfall__daily_depth'] = np.ones(grid.number_of_cells)

rad = Radiation(grid, method='Grid', clearsky_turbidity=1.0, opt_airmass=2.0)
PET = PotentialEvapotranspiration(grid, method='PriestleyTaylor')
SM  = SoilMoisture(grid, runon=0, f_bare=0.7)


## 9. Daily Water Balance Time Loop

Each iteration of the loop processes one day in the event window:

1. **Runoff abstraction (SCS curve number)**: surface runoff is estimated using the
   SCS-CN method. Maximum soil storage `Smax = (Porosity − θ) × 400` is computed
   from current soil moisture `θ` and porosity. Initial abstraction is 5% of `Smax`.
   For vegetated cells (PFT ≠ 3), a fixed curve number CN = 45 is applied. Runoff
   is set to zero where `P ≤ Initial_abstraction`.

2. **Effective precipitation**: `Peffective = P − Initial_abstraction − Runoff`.
   This is the rainfall available to infiltrate and recharge the root zone.

3. **Radiation**: `Radiation.update()` computes net shortwave and longwave radiation
   at each cell from Tmin, Tmax, latitude, albedo, and day-of-year.

4. **PET**: `PotentialEvapotranspiration.update()` applies the Priestley-Taylor
   equation to the radiation output and temperature fields.

5. **Soil moisture**: `SoilMoisture.update()` advances the root-zone water balance
   by one day, consuming `Peffective` and `PET` and producing `soil_moisture__root_zone_leakage`
   (recharge, mm/day) and `surface__evapotranspiration` (AET, mm/day).

6. **Cell-to-node mapping**: leakage, runoff, soil moisture, and AET are read from
   cell arrays and written to node arrays for output and later raster export.

`current_time` is advanced by the component at the end of each iteration.


In [9]:
for idx, (rainfall, tempmin, tempmax) in enumerate(
        zip(rainfall_arrays, tempmin_arrays, tempmax_arrays)):

    # ── Load daily forcings ───────────────────────────────────────────────────
    _ = grid.add_field('Precipitation', rainfall, at='node', clobber=True)
    grid.set_nodata_nodes_to_closed(rainfall, -9999.)
    _ = grid.add_field('Tmin', tempmin, at='node', clobber=True)
    grid.set_nodata_nodes_to_closed(tempmin, -9999.)
    _ = grid.add_field('Tmax', tempmax, at='node', clobber=True)
    grid.set_nodata_nodes_to_closed(tempmax, -9999.)

    # ── SCS-CN runoff abstraction ─────────────────────────────────────────────
    grid['cell']['Precipitation'] = map_node_to_cell(grid, 'Precipitation')
    P = grid['cell']['Precipitation']

    Soil_Saturation = grid['cell']['soil_moisture__saturation_fraction']
    Soil_moisture   = Soil_Saturation * Porosity
    Smax = (Porosity - Soil_moisture) * 400
    Initial_abstraction = 0.05 * Smax

    grid['cell']['Runoff'] = ((P - Initial_abstraction) ** 2) / (P - Initial_abstraction + Smax)

    # Fixed CN = 45 for cells with PFT class 3 (e.g. shrubland)
    pft3 = grid.at_cell['vegetation__plant_functional_type'] == 3
    grid.at_cell['Runoff'][pft3] = (
        (grid.at_cell['Precipitation'][pft3] - 0.1 * 45) ** 2 /
        (grid.at_cell['Precipitation'][pft3] - 0.1 * 45 + 45)
    )
    grid.at_cell['Runoff'][P - Initial_abstraction <= 0] = 0

    Peffective = P - Initial_abstraction - grid['cell']['Runoff']
    grid['cell']['rainfall__daily_depth'] = Peffective

    # ── Radiation ─────────────────────────────────────────────────────────────
    rad._current_time = current_time
    rad._latitude     = Latitude
    rad._A            = Albedo
    rad._Tmin         = grid.at_node['Tmin']
    rad._Tmax         = grid.at_node['Tmax']
    rad.update()

    # ── PET ───────────────────────────────────────────────────────────────────
    PET._current_time = current_time
    PET._latitude     = Latitude
    PET._a            = Albedo
    PET._Tmin         = grid.at_node['Tmin']
    PET._Tmax         = grid.at_node['Tmax']
    PET.update()

    # ── Soil moisture water balance ───────────────────────────────────────────
    SM._current_time = current_time
    SM._Tb = 24   # storm duration (hours); daily time step
    SM.update()

    # ── Append regional averages (diagnostic) ────────────────────────────────
    P_REG.append(np.mean(grid.at_node['Precipitation'][grid.core_nodes]))
    SM_REG.append(np.mean(grid['cell']['soil_moisture__saturation_fraction'] * Porosity))
    RE_REG.append(np.mean(grid.at_cell['soil_moisture__root_zone_leakage']))
    PET_REG.append(np.mean(grid.at_cell['surface__potential_evapotranspiration_rate']))
    AET_REG.append(np.mean(grid.at_cell['surface__evapotranspiration']))

    # ── Map cell outputs to nodes ─────────────────────────────────────────────
    leakage_to_node       = np.zeros(grid.number_of_nodes)
    runoff_to_node        = np.zeros(grid.number_of_nodes)
    soil_moisture_to_node = np.zeros(grid.number_of_nodes)
    ET_to_node            = np.zeros(grid.number_of_nodes)

    for cell in range(grid.number_of_cells):
        node_id = grid.node_at_cell[cell]
        leakage_to_node[node_id]       = grid.at_cell['soil_moisture__root_zone_leakage'][cell]
        runoff_to_node[node_id]        = grid.at_cell['Runoff'][cell]
        soil_moisture_to_node[node_id] = grid.at_cell['soil_moisture__saturation_fraction'][cell]
        ET_to_node[node_id]            = grid.at_cell['surface__evapotranspiration'][cell]

    recharge_arrays.append(leakage_to_node)
    runoff_arrays.append(runoff_to_node)
    soil_moisture_arrays.append(soil_moisture_to_node)
    ET_arrays.append(ET_to_node)

    current_time = SM.current_time

    print(f"Day {idx + 1}: mean P = {P_REG[-1]:.2f} mm, "
          f"mean recharge = {RE_REG[-1]:.2f} mm, "
          f"mean SM = {SM_REG[-1]:.3f} m3/m3")


Day 1: mean P = 0.58 mm, mean recharge = 0.00 mm, mean SM = 0.241 m3/m3
Day 2: mean P = 0.39 mm, mean recharge = 0.00 mm, mean SM = 0.236 m3/m3
Day 3: mean P = 2.05 mm, mean recharge = 0.00 mm, mean SM = 0.231 m3/m3
Day 4: mean P = 10.59 mm, mean recharge = 0.01 mm, mean SM = 0.243 m3/m3
Day 5: mean P = 38.23 mm, mean recharge = 1.40 mm, mean SM = 0.302 m3/m3
Day 6: mean P = 29.85 mm, mean recharge = 6.27 mm, mean SM = 0.330 m3/m3
Day 7: mean P = 0.31 mm, mean recharge = 1.97 mm, mean SM = 0.317 m3/m3


## 10. Recharge Routing and Output

### 10a. Peak recharge composite

The peak-event recharge grid is first computed as the cell-wise maximum daily recharge
across the event window. This represents the single-day maximum infiltration at each
node — the most critical soil moisture state for slope stability.

### 10b. Flow-accumulation routing

Raw recharge from the soil moisture model is a local, cell-scale quantity. To better
represent the upslope moisture loading that drives groundwater pressure at a given
node — and to match the assumptions of the `LandslideProbability` component's
steady-state wetness index — recharge is routed downstream using `FlowAccumulator`.

The routing procedure:

1. `FlowAccumulator` (D8) accumulates `water__unit_flux_in` (the peak recharge field)
   downstream, producing `surface_water__discharge` (units: mm × m²).
2. Routed recharge = `surface_water__discharge / drainage_area`, yielding an
   upslope-area-weighted mean recharge in mm/day at each node. Nodes with zero drainage
   area are assigned one cell area (`dx × dy`) to avoid division by zero.

The routed recharge grid is written to `output_folder` as an ESRI ASCII grid. This
file is used directly as input to the `LandslideProbability` notebook.

> **For historical composite runs** (Scenario 2 in the landslide notebook), repeat this
> notebook for each historical event and save each routed output. The cell-wise maximum
> across events is then computed in the landslide notebook.


In [10]:
# ── 10a. Peak recharge: cell-wise maximum across event window ────────────────
peak_recharge = np.maximum.reduce(recharge_arrays)

# Replace any non-positive values with a small positive floor before routing
for i in range(len(peak_recharge)):
    if peak_recharge[i] <= 0:
        peak_recharge[i] = 0.0001

_ = grid.add_field('water__unit_flux_in', peak_recharge, at='node', clobber=True)

print(f"Pre-routing recharge — mean (core): {peak_recharge[grid.core_nodes].mean():.4f} mm")
print(f"Pre-routing recharge — max  (core): {peak_recharge[grid.core_nodes].max():.4f} mm")

# ── 10b. Flow-accumulation routing ───────────────────────────────────────────
fa_routed = FlowAccumulator(grid,
                             surface='topographic__elevation',
                             flow_director='FlowDirectorD8',
                             runoff_rate='water__unit_flux_in')
(da_routed, q_routed) = fa_routed.accumulate_flow()

# Prevent division by zero: assign one cell area to any zero-drainage-area nodes
da_routed[da_routed == 0] = grid.dx * grid.dy

# Routed recharge = accumulated discharge / drainage area (mm/day, upslope-averaged)
routed_recharge = grid.at_node['surface_water__discharge'] / da_routed
_ = grid.add_field('routed_recharge', routed_recharge, at='node', clobber=True)

print(f"Routed recharge     — mean (core): {routed_recharge[grid.core_nodes].mean():.4f} mm")
print(f"Routed recharge     — max  (core): {routed_recharge[grid.core_nodes].max():.4f} mm")

# ── Save output ───────────────────────────────────────────────────────────────
event_label = 'March_2023'   # update this label when modeling a different event
output_filename = output_folder + f'IMERG_Recharge_{event_label}_routed.asc'

write_esri_ascii(output_filename, grid, 'routed_recharge', clobber=True)


Pre-routing recharge — mean (core): 6.1239 mm
Pre-routing recharge — max  (core): 27.7999 mm
FlowAccumulator found both the field 'water__unit_flux_in' and a provided float or array for the runoff_rate argument. THE FIELD IS BEING OVERWRITTEN WITH THE SUPPLIED RUNOFF_RATE!
Routed recharge     — mean (core): 6.1283 mm
Routed recharge     — max  (core): 27.3474 mm


['recharge/IMERG_Recharge_March_2023_routed.asc']

## 11. Run Time

In [11]:
elapsed = time.time() - st
print(f"Total elapsed time: {elapsed:.1f} s  ({elapsed / 60:.1f} min)")


Total elapsed time: 420.4 s  (7.0 min)
